In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Riyaz\\OneDrive\\Desktop\\text_summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd


'c:\\Users\\Riyaz\\OneDrive\\Desktop\\text_summarizer'

In [5]:
## this is entity class
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class  DataValidationConfig:
    root_dir :Path
    data_path : Path
    model_ckpt:Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evalution_strategy: str
    eval_steps : int
    save_steps: float
    gradient_saccumulation_steps : int
    

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories
from textSummarizer.entity import ModelTrainerConfig

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath= PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params= read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config =self.config.model_trainer
        params= self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.evaluation_strategy,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )

        return model_trainer_config

In [8]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

c:\Users\Riyaz\OneDrive\Desktop\text_summarizer\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
def train(self):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(
        "t5-small"
    )

    model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
        "t5-small"
    ).to(device)

    seq2seq_data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model_pegasus
    )

    # loading data
    dataset_samsum_pt = load_from_disk(
        self.config.data_path
    )

    # VERY SMALL TRAINING DATA
    small_train_dataset = dataset_samsum_pt["train"].select(range(10))
    small_eval_dataset = dataset_samsum_pt["validation"].select(range(2))

    trainer_args = TrainingArguments(
        output_dir=self.config.root_dir,

        num_train_epochs=1,

        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,

        logging_steps=1,

        save_steps=5,

        gradient_accumulation_steps=1
    )

    trainer = Trainer(
        model=model_pegasus,
        args=trainer_args,
        data_collator=seq2seq_data_collator,
        train_dataset=small_train_dataset,
        eval_dataset=small_eval_dataset
    )

    trainer.train()

    model_pegasus.save_pretrained(
        os.path.join(
            self.config.root_dir,
            "t5-small-model"
        )
    )

    tokenizer.save_pretrained(
        os.path.join(
            self.config.root_dir,
            "tokenizer"
        )
    )

In [10]:
from textSummarizer.config.configuration import ConfigurationManager

config = ConfigurationManager()

print(config.params.TrainingArguments)

[2026-05-11 21:33:10,544: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 21:33:10,550: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 21:33:10,553: INFO: common: created directory at: artifacts]
{'num_train_epochs': 1, 'warmup_steps': 500, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 1, 'weight_decay': 0.01, 'logging_steps': 10, 'evaluation_strategy': 'steps', 'eval_steps': 500, 'save_steps': 1000000, 'gradient_accumulation_steps': 16}


In [11]:
print(config.params.TrainingArguments)

{'num_train_epochs': 1, 'warmup_steps': 500, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 1, 'weight_decay': 0.01, 'logging_steps': 10, 'evaluation_strategy': 'steps', 'eval_steps': 500, 'save_steps': 1000000, 'gradient_accumulation_steps': 16}


In [12]:
from textSummarizer.entity import ModelTrainerConfig

help(ModelTrainerConfig)

Help on class ModelTrainerConfig in module textSummarizer.entity:

class ModelTrainerConfig(builtins.object)
 |  ModelTrainerConfig(
 |      root_dir: pathlib._local.Path,
 |      data_path: pathlib._local.Path,
 |      model_ckpt: pathlib._local.Path,
 |      num_train_epochs: int,
 |      warmup_steps: int,
 |      per_device_train_batch_size: int,
 |      per_device_eval_batch_size: int,
 |      weight_decay: float,
 |      logging_steps: int,
 |      evaluation_strategy: str,
 |      eval_steps: int,
 |      save_steps: int,
 |      gradient_accumulation_steps: int
 |  ) -> None
 |
 |  ModelTrainerConfig(root_dir: pathlib._local.Path, data_path: pathlib._local.Path, model_ckpt: pathlib._local.Path, num_train_epochs: int, warmup_steps: int, per_device_train_batch_size: int, per_device_eval_batch_size: int, weight_decay: float, logging_steps: int, evaluation_strategy: str, eval_steps: int, save_steps: int, gradient_accumulation_steps: int)
 |
 |  Methods defined here:
 |
 |  __eq__(s

In [16]:
from textSummarizer.components.model_trainer import ModelTrainer

In [17]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer_config = ModelTrainer(
        config=model_trainer_config
    )

    model_trainer_config.train()

except Exception as e:
    raise e

[2026-05-11 21:34:59,184: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 21:34:59,189: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 21:34:59,191: INFO: common: created directory at: artifacts]
[2026-05-11 21:34:59,195: INFO: common: created directory at: artifacts/model_trainer]
Using device: cpu
[2026-05-11 21:35:00,908: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 3748.39it/s]


[2026-05-11 21:35:01,739: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/generation_config.json "HTTP/1.1 200 OK"]
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})


c:\Users\Riyaz\OneDrive\Desktop\text_summarizer\textS\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,3.624394
2,2.714494
3,2.753577
4,3.293577
5,2.893414
6,2.944897
7,4.879562
8,3.753808
9,3.349591
10,3.524504


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.34it/s]

Model training completed successfully
